# CorticalLIME: Explaining a Biomimetic Auditory Network

This notebook demonstrates **CorticalLIME**, a perturbation-based explainer that operates directly on the *biological* dimensions of the diffAudNeuro frontend (Famularo et al. 2024, [arXiv:2409.08997](https://arxiv.org/abs/2409.08997)).

## Why perturb (Rate $\omega$, Scale $\Omega$) instead of (Time, Frequency)?

Standard audio LIME variants ([SoundLIME](https://github.com/saum25/SoundLIME), audioLIME, ...) mask rectangular patches of the time-frequency plane. Those patches have **no biological referent**: a block of spectrogram pixels does not correspond to any neural population in primary auditory cortex (A1).

The diffAudNeuro frontend instead expands each auditory spectrogram into a bank of $N$ STRF channels indexed by $(\Omega, \omega)$ pairs (spectral scale in cyc/oct, temporal rate in Hz). Each channel is a direct analogue of a modulation-tuned A1 neuron (Chi, Ru \& Shamma, 2005). Masking an entire $(\Omega, \omega)$ channel therefore *lesions* a biologically meaningful population across the whole utterance and tells us **which modulation channels the downstream classifier is actually using to discriminate phonemes**.

## Pipeline

1. Encode audio once with the trained `vSupervisedSTRF` frontend $\rightarrow$ 4D cortical features `(F=128, T=200, S=40)`.
2. Draw $N$ binary masks over the $S=40$ STRF channels.
3. Multiply each mask along the channel axis and run the **frozen CNN decoder**.
4. Read off the predicted-class probability for each perturbation.
5. Fit a weighted Ridge surrogate $\text{mask} \to P(\text{class})$.
6. Visualise the per-channel coefficients on the $(\Omega, \omega)$ plane.

## 1. Environment setup (Colab)

Skip the install cell when running locally with the dependencies already present.

In [ ]:
import os, sys, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install 'jax[cuda12]' flax optax librosa kagglehub scikit-learn editdistance soundfile
    if not os.path.isdir('/content/Cortical_Front'):
        !git clone -q https://github.com/MeysamAmirsardari/Cortical_Front /content/Cortical_Front
    REPO = '/content/Cortical_Front'
else:
    # Worktree path for local development.
    REPO = os.environ.get('CORTICAL_REPO', '/Users/eminent/Projects/Cortical_Front')
R_CODE = os.path.join(REPO, 'r_code')
WORKTREE = os.path.join(REPO, '.claude/worktrees/cranky-kowalevski') if not IN_COLAB else REPO
sys.path.insert(0, R_CODE)
sys.path.insert(0, WORKTREE)
os.chdir(R_CODE)  # supervisedSTRF loads cochlear_filter_params.npz from CWD
print('Repo:', REPO)

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pickle, glob, re

plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('jax', jax.__version__, '| devices:', jax.devices())

## 2. Build the cochlear filter cache and load the trained checkpoint

In [ ]:
# Make sure cochlear_filter_params.npz exists (supervisedSTRF loads it from CWD).
if not Path('cochlear_filter_params.npz').is_file():
    from strfpy_jax import read_cochba_j
    Bs, As = read_cochba_j()
    np.savez('cochlear_filter_params.npz', Bs=np.asarray(Bs), As=np.asarray(As))
    print('Built cochlear_filter_params.npz from cochba.txt')

from supervisedSTRF import vSupervisedSTRF
from cortical_lime import CorticalLIME, make_jax_callables

# Locate the latest TIMIT phoneme checkpoint.
MODEL_DIR = Path(os.environ.get('MODEL_DIR', 'nishit_trained_models/main_jax_phoneme_rec_timit'))
if not MODEL_DIR.is_dir():
    MODEL_DIR = Path('models/main_jax_phoneme_rec_timit')
ckpts = sorted(
    glob.glob(str(MODEL_DIR / 'chkStep_*.p')),
    key=lambda p: int(re.search(r'chkStep_(\d+)\.p$', p).group(1)),
)
assert ckpts, f'No chkStep_*.p found in {MODEL_DIR}'
CKPT = ckpts[-1]
print('Checkpoint:', CKPT)

with open(CKPT, 'rb') as f:
    obj = pickle.load(f)
nn_params, aud_params = obj if isinstance(obj, (list, tuple)) else (obj['nn_params'], obj['params'])

sr_pairs = np.asarray(aud_params['sr'])  # (n_strfs, 2) -> (scale, rate)
N_STRFS = sr_pairs.shape[0]
print(f'STRF bank: {N_STRFS} channels')
print(f'Scales (cyc/oct): min={sr_pairs[:,0].min():.2f}  max={sr_pairs[:,0].max():.2f}')
print(f'Rates (Hz):       min={sr_pairs[:,1].min():.2f}  max={sr_pairs[:,1].max():.2f}')

In [ ]:
# Build the model with the same hyper-parameters used at training time.
model = vSupervisedSTRF(
    n_phones=61,
    input_type='audio',
    update_lin=True,
    use_class=False,
    encoder_type='strf',
    decoder_type='cnn',
    compression_method='power',
    conv_feats=[10, 20, 40],
    pooling_stride=2,
)
encode_fn, decode_fn = make_jax_callables(model, nn_params, aud_params)

# Sanity-check shapes with a dummy 1-second waveform.
_dummy = np.zeros((1, 16000), dtype=np.float32)
_feats = encode_fn(_dummy)
_logits = decode_fn(_feats)
print('cortical features:', _feats.shape, '   decoder logits:', _logits.shape)

## 3. Pick a TIMIT example

Reuse the Kaggle TIMIT mirror used by `evaluate_timit_colab.ipynb`. If running offline, drop your own 16 kHz mono WAV next to this notebook and point `WAV` at it.

In [ ]:
import librosa
WAV = os.environ.get('CORTICAL_LIME_WAV')
if WAV is None:
    try:
        import kagglehub
        timit_root = Path(kagglehub.dataset_download('mfekadu/darpa-timit-acousticphonetic-continuous-speech'))
        wavs = sorted(timit_root.rglob('TEST/**/*.[Ww][Aa][Vv]'))
        assert wavs, 'No TEST WAVs found in Kaggle TIMIT mirror'
        WAV = str(wavs[42])  # arbitrary stable pick
    except Exception as e:
        print('Kaggle download failed:', e)
        raise
print('WAV:', WAV)

y, _ = librosa.load(WAV, sr=16000, mono=True)
if len(y) >= 16000:
    y = y[(len(y) - 16000) // 2 : (len(y) - 16000) // 2 + 16000]
else:
    y = np.pad(y, (0, 16000 - len(y)))
rms = np.sqrt(np.mean(y ** 2))
if rms > 0:
    y = (y / rms).astype(np.float32)
print('waveform:', y.shape, 'rms-normalised')

## 4. Run CorticalLIME

In [ ]:
explainer = CorticalLIME(
    encode_fn=encode_fn,
    decode_fn=decode_fn,
    sr=sr_pairs,
    n_samples=600,
    keep_prob=0.5,
    distance_metric='cosine',
    kernel_width=0.25,
    ridge_alpha=1.0,
    batch_size=32,
    seed=0,
)
result = explainer.explain(y)

TIMIT_PHONEMES = [
    'aa','ae','ah','ao','aw','ax','ax-h','axr','ay','b','bcl','ch','d','dcl','dh','dx',
    'eh','el','em','en','eng','epi','er','ey','f','g','gcl','h#','hh','hv','ih','ix','iy',
    'jh','k','kcl','l','m','n','ng','nx','ow','oy','p','pau','pcl','q','r','s','sh','t',
    'tcl','th','uh','uw','ux','v','w','y','z','zh',
]
def idx2phn(i):
    return '<blank>' if i == 0 else (TIMIT_PHONEMES[i-1] if 1 <= i <= 61 else f'<{i}>')

print(f'Target class: {result.target_class}  ({idx2phn(result.target_class)})')
print(f'Reference P(class) = {result.target_prob:.4f}')
print(f'Surrogate Ridge R^2 = {result.surrogate_r2:.4f}')
print('\nTop-5 most important STRF channels (|coef|):')
for ch, s, r, w in result.top_k(5):
    print(f'  ch={ch:2d}  scale={s:+.2f} cyc/oct  rate={r:+6.2f} Hz   coef={w:+.4f}')

## 5. Publication-ready figures

In [ ]:
# --- Figure 1: input auditory spectrogram + cortical channel energies -----
feats = encode_fn(y[None, :])[0]               # (F, T, S)
audspec_proxy = feats.sum(axis=-1)              # collapse STRFs to view F x T
channel_energy = (feats ** 2).mean(axis=(0, 1)) # (S,)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={'width_ratios': [2, 1]})
im = axes[0].imshow(audspec_proxy, origin='lower', aspect='auto', cmap='magma')
axes[0].set(title='Cortical features collapsed over STRFs', xlabel='Time frame', ylabel='Cochlear channel')
fig.colorbar(im, ax=axes[0], shrink=0.85)

order = np.argsort(channel_energy)[::-1]
axes[1].bar(range(len(channel_energy)), channel_energy[order], color='#444')
axes[1].set(title='Per-STRF mean energy', xlabel='STRF rank', ylabel='Energy')
fig.tight_layout()

In [ ]:
# --- Figure 2: importance scatter on the (rate, scale) plane --------------
imp = result.importances
vmax = np.max(np.abs(imp)) + 1e-12

fig, ax = plt.subplots(figsize=(6.4, 4.6))
sc = ax.scatter(
    sr_pairs[:, 1], sr_pairs[:, 0],
    c=imp, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
    s=140 + 600 * (np.abs(imp) / vmax),
    edgecolors='black', linewidths=0.6,
)
ax.axvline(0, color='gray', lw=0.6, ls=':')
ax.set(
    xlabel=r'Temporal rate  $\omega$  (Hz; sign = up/down sweep)',
    ylabel=r'Spectral scale  $\Omega$  (cyc / oct)',
    title=f'CorticalLIME: STRF importance for class "{idx2phn(result.target_class)}"',
)
cb = fig.colorbar(sc, ax=ax)
cb.set_label('Ridge coefficient (\u2192 P(class))')
fig.tight_layout()

In [ ]:
# --- Figure 3: binned (Scale x Rate) importance grid -----------------------
grid, rate_edges, scale_edges = result.rate_scale_grid(n_rate_bins=10, n_scale_bins=6)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
im = ax.imshow(
    grid, origin='lower', aspect='auto', cmap='RdBu_r',
    vmin=-np.nanmax(np.abs(grid)), vmax=np.nanmax(np.abs(grid)),
    extent=[rate_edges[0], rate_edges[-1], scale_edges[0], scale_edges[-1]],
)
ax.set(
    xlabel=r'Temporal rate  $\omega$  (Hz)',
    ylabel=r'Spectral scale  $\Omega$  (cyc / oct)',
    title='Mean Ridge coefficient per (\u03A9, \u03C9) bin',
)
fig.colorbar(im, ax=ax, label='mean coef')
fig.tight_layout()

In [ ]:
# --- Figure 4: surrogate diagnostic ---------------------------------------
y_true = result.target_probs
y_pred = result.masks @ result.importances + result.intercept
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

axes[0].scatter(y_pred, y_true, c=result.weights, cmap='viridis', s=12)
lims = [min(y_pred.min(), y_true.min()), max(y_pred.max(), y_true.max())]
axes[0].plot(lims, lims, 'k--', lw=0.8)
axes[0].set(xlabel='Ridge prediction', ylabel='Decoder P(class)',
            title=f'Surrogate fidelity  (R\u00b2 = {result.surrogate_r2:.3f})')

axes[1].hist(result.distances, bins=30, color='#666')
axes[1].set(xlabel='cosine distance to all-ones mask', ylabel='count',
            title='Perturbation distance distribution')
fig.tight_layout()

## 6. Sanity check: ablate the top STRFs

If CorticalLIME is faithful, zeroing the channels it flagged as positive contributors should drop the predicted-class probability much more than zeroing random channels of the same count (a single-point deletion test in the spirit of the Faithfulness pillar of the proposal).

In [ ]:
k = 5
feats = encode_fn(y[None, :])
ref = explainer._utterance_class_probs(decode_fn(feats))[0, result.target_class]

top_pos = np.argsort(-result.importances)[:k]
mask = np.ones(N_STRFS, dtype=np.float32); mask[top_pos] = 0.0
ablated = feats * mask[None, None, None, :]
p_top = explainer._utterance_class_probs(decode_fn(ablated))[0, result.target_class]

rng = np.random.default_rng(1)
rand_drops = []
for _ in range(20):
    idx = rng.choice(N_STRFS, size=k, replace=False)
    m = np.ones(N_STRFS, dtype=np.float32); m[idx] = 0.0
    rand_drops.append(
        float(explainer._utterance_class_probs(decode_fn(feats * m[None, None, None, :]))[0, result.target_class])
    )
rand_drops = np.array(rand_drops)

print(f'Reference P(class)            : {ref:.4f}')
print(f'Ablate top-{k} CorticalLIME    : {p_top:.4f}   (drop {ref-p_top:+.4f})')
print(f'Ablate {k} random STRFs (mean) : {rand_drops.mean():.4f}   (drop {ref-rand_drops.mean():+.4f})')

fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.hist(ref - rand_drops, bins=12, color='#999', label='random ablation')
ax.axvline(ref - p_top, color='#c0392b', lw=2.2, label=f'top-{k} CorticalLIME')
ax.set(xlabel='\u0394 P(class)  (reference - ablated)', ylabel='count',
       title='Faithfulness check: deletion of k=5 STRFs')
ax.legend(frameon=False)
fig.tight_layout()

## Notes

- The CorticalLIME logic lives in `cortical_lime.py` and is intentionally framework-agnostic: `perturb_cortical_features`, `mask_distance`, `kernel_weights`, and `fit_surrogate` are pure NumPy / scikit-learn helpers; the JAX-specific binding is isolated in `make_jax_callables`.
- The forward pass uses `torch.no_grad()`-equivalent semantics: the JAX callables are wrapped in `@jax.jit` and never request gradients, so the model is treated as a frozen function.
- To extend the analysis to multiple utterances or to compare English vs. Spanish models (Faithfulness Evaluation pillar of the proposal), call `explainer.explain(...)` per utterance and average the resulting `importances` arrays.